In [ ]:
# README:

# This Jupyter notebook will convert .nemo PnC models to .onnx and will additionally shrink those .onnx files
#
# Possible prerequisites:
# ========================
# !pip install causal-conv1d
# BRANCH = "r2.4.0"
# !pip install torch==2.4.0
# !pip install causal-conv1d>=1.4.0
# !pip install torch
# !pip install "nemo_toolkit[all] @ git+https://github.com/NVIDIA/NeMo.git@$BRANCH"
#
# Additionally, for shrinking of onnx models sizes RUN THE FOLLOWING COMMAND
# !pip install onnx-shrink-ray==0.0.9
#
# IMPORTANT INFO:
# Afterwards, when converting a bug may appear in form of
#           shrink.py", line 36, in gather_initializers_in_graph
#           graph.initializer.clear()
#           AttributeError: 'google._upb._message.RepeatedCompositeContainer' object has no attribute 'clear'
# To solve you may need to open
#     miniconda3/envs/your_conda_env/lib/python3.10/site-packages/onnx_shrink_ray/shrink.py
# and REPLACE LINE 36, which is
#       graph.initializer.clear()
# WITH the following
#       del graph.initializer[:]

In [ ]:
import io
import os
import nemo
from nemo.collections.nlp.models import PunctuationCapitalizationModel

device = "cpu"

base_path = "/mnt/e/pnc"
base_path_nemo = os.path.join(base_path, "nemo_files")
base_path_out = os.path.join(base_path, "final")
nemo_model_files = []

In [ ]:
# export PnC model files into "/final" folder as .onnx files

files = os.listdir(base_path_nemo)
for filename in files:    
    full_filename = os.path.join(base_path_nemo, filename)
    outname = os.path.splitext(filename)[0] + ".onnx"
    onnx_path = os.path.join(base_path_out, outname)
    model = PunctuationCapitalizationModel.restore_from(full_filename, map_location=device)
    model.export(onnx_path)

In [ ]:
for file in os.listdir(base_path_out):
    full_path_in = os.path.join(base_path_out, file)
    !python -m onnx_shrink_ray.shrink {full_path_in}
    print("Shrinked file ", file)

After you have ran all of the code above, the PnC onnx models that are used in the project<br>
are those in /final/*_quantized_weights.onnx